# 03_9 선택 모델 운영 적용

사용자 확정: **5일 LightGBM+DLinear, 20일 LightGBM+XGBoost 가격 단순평균, 60일 DLinear**. 모두 가격·기후·거시경제와 조건부 뉴스 feature를 사용한다. Naive는 구성원에 포함하지 않는다.

03_8의 Validation 선택 설정과 2025-12-31까지 학습한 가중치를 그대로 내보낸다. 추가 탐색·재학습 없이 적용하며 기존 연구 결과를 덮어쓰지 않는다. 2026년은 이미 확인한 역사적 평가 기간으로, 미사용 Test가 아니다.

뉴스는 유일한 최댓값이 bullish면 +1, bearish면 -1, neutral/uncertain/동률이면 0에 relevance×confidence를 곱한다. 일별 합을 tanh로 압축한다. 뉴스가 없거나 일별 합이 0이면 같은 설정의 27개 기본 feature 모델로 복귀한다. 실시간 운용은 실제 분석 결과가 이용 가능해진 첫 거래일의 UTC 23시 기준으로 배정한다. 분석 지연이 7일을 넘으면 오래된 소급 분석으로 보고 운영 입력에서 제외한다. 사후 분류된 과거 기사를 오늘의 새 뉴스로 다시 넣지 않는다. 따라서 아래 실제 가용 시각 결과는 03_8의 과거 기사 재분류 연구 결과와 다를 수 있다.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'coffee_service').exists() else Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import pandas as pd
from datetime import date
from data_code.export_selected_models import export_selected, RESEARCH_RUN
from coffee_service.selected_models import (SelectedBundle, SELECTION, assemble_selected, daily_news,
                                           load_weather, selected_predictions)
from coffee_service.pipeline import sources_as_of
from coffee_service import jev_store
artifact = export_selected()
bundle = SelectedBundle(artifact)
pd.DataFrame([{'지평 (거래일)': h, '선택 모델': ' + '.join(f'{name}@{setting}' for name, setting in members),
               '결합 방식': '가격 단순평균' if len(members)>1 else '단일 모델', '뉴스 feature': '조건부 사용',
               '학습 종료': bundle.manifest['train_cutoff'], '모델 ID': bundle.model_id(h)}
              for h, members in SELECTION.items()])

,지평 (거래일),선택 모델,결합 방식,뉴스 feature,학습 종료,모델 ID
0,5,LightGBM@300 + DLinear@10,가격 단순평균,조건부 사용,2025-12-31,lightgbm-dlinear-news-h5-b47bac4ae518
1,20,LightGBM@100 + XGBoost@100,가격 단순평균,조건부 사용,2025-12-31,lightgbm-xgboost-news-h20-b47bac4ae518
2,60,DLinear@30,단일 모델,조건부 사용,2025-12-31,dlinear-news-h60-b47bac4ae518


In [2]:
source_dir = ROOT / 'data/processed/fixed_ensemble_news/sources_20260928'
end = date(2026, 9, 25)  # 재현 가능한 원천 snapshot 종료일
sources = sources_as_of(source_dir, end)
dataset = assemble_selected(load_weather(sources, source_dir, end), bundle.manifest['train_cutoff'])
records = jev_store.read_analyses(ROOT/'data/jev', latest=True)
news = daily_news(records, dataset.sessions, as_of='2026-09-28T23:00:00Z')
predictions = selected_predictions(dataset, bundle, news)
latest = predictions[predictions.origin_date.eq(predictions.origin_date.max())]
assert set(latest.horizon) == {5,20,60}
assert not predictions.model_id.str.contains('persistence', case=False).any()
latest[['horizon','origin_date','target_date','predicted_price','final_direction','news_article_count','signal_status']].rename(columns={
    'horizon':'지평 (거래일)','origin_date':'기준일','target_date':'목표일','predicted_price':'예측가 (¢/lb)',
    'final_direction':'가격 방향','news_article_count':'이용 가능 기사 수','signal_status':'뉴스 적용 상태'})

,지평 (거래일),기준일,목표일,예측가 (¢/lb),가격 방향,이용 가능 기사 수,뉴스 적용 상태
549,5,2026-09-25,2026-10-02,278.213619,DOWN,0,numeric_fallback
550,20,2026-09-25,2026-10-23,285.326050,UP,0,numeric_fallback
551,60,2026-09-25,2026-12-21,362.728512,UP,0,numeric_fallback


## 실험 결과와 운영 추론 대조

각 구성원의 base/news 가중치와 전처리 통계는 해시로 검사한다. 다음 검사는 03_8의 동일 입력에서 5·20·60일 가격 평균을 재현한다. 최신 운영 예측은 미성숙 목표일도 생성하며, DB는 모델 버전별 기존 예측을 보존하고 실제 가격만 채운다. 별도 뉴스 잔차 보정은 중복 적용하지 않는다.

In [3]:
from data_code.fixed_ensemble_news import load_market
from data_code.news_expanding_benchmark import prepare_fold
import numpy as np
sources, sessions, _ = load_market(source_dir)
research_news = pd.read_parquet(ROOT/'data/processed/news_feature_ensemble/ed374e9c9c0a19c6/daily_news.parquet')
checks = []
for horizon, members in SELECTION.items():
    fold = prepare_fold(sources, sessions, bundle.columns[:-1], research_news.set_index("date").news_sentiment,
                        horizon, '2025-12-31', end)
    meta = fold['metadata']
    windows = fold['eval_x'].astype(float)*meta['input_scale'] + meta['input_mean']
    expected = []
    for name, setting in members:
        saved = pd.read_parquet(RESEARCH_RUN/f'models/evaluation_2026_{horizon}/{name}/predictions.parquet')
        saved = saved[saved.setting.eq(setting)].sort_values('origin_date')
        assert list(saved.origin_date) == list(fold['reference'].origin_date)
        expected.append(saved.predicted_return.to_numpy())
    expected = np.logaddexp.reduce(np.stack(expected),axis=0)-np.log(len(members))
    actual = bundle.predict(horizon, windows)
    np.testing.assert_allclose(actual, expected, rtol=1e-6, atol=1e-7)
    checks.append({'지평 (거래일)':horizon, '대조 표본':len(actual), '최대 수익률 차이':float(np.max(np.abs(actual-expected))),
                   '검증':'통과'})
pd.DataFrame(checks)

,지평 (거래일),대조 표본,최대 수익률 차이,검증
0,5,179,1.856159e-09,통과
1,20,164,0.000000e+00,통과
2,60,124,0.000000e+00,통과
